In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

In [2]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\LENOVO\Desktop\nifty100_project")

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"

ETL_DIR = BASE_DIR / "etl"
NOTEBOOK_DIR = BASE_DIR / "notebook"
SQL_DIR = BASE_DIR / "sql"
DOCX_DIR = BASE_DIR / "docx"

print("BASE     :", BASE_DIR)
print("RAW      :", RAW_DIR)
print("CLEAN    :", CLEAN_DIR)
print("ETL      :", ETL_DIR)
print("NOTEBOOK :", NOTEBOOK_DIR)
print("SQL      :", SQL_DIR)
print("DOCX     :", DOCX_DIR)

BASE     : C:\Users\LENOVO\Desktop\nifty100_project
RAW      : C:\Users\LENOVO\Desktop\nifty100_project\data\raw
CLEAN    : C:\Users\LENOVO\Desktop\nifty100_project\data\clean
ETL      : C:\Users\LENOVO\Desktop\nifty100_project\etl
NOTEBOOK : C:\Users\LENOVO\Desktop\nifty100_project\notebook
SQL      : C:\Users\LENOVO\Desktop\nifty100_project\sql
DOCX     : C:\Users\LENOVO\Desktop\nifty100_project\docx


In [3]:
for folder in [DATA_DIR, RAW_DIR, CLEAN_DIR, ETL_DIR, NOTEBOOK_DIR, SQL_DIR, DOCX_DIR]:
    print(folder.name, "->", folder.exists())

data -> True
raw -> True
clean -> True
etl -> True
notebook -> True
sql -> True
docx -> True


In [4]:
list(RAW_DIR.glob("*"))

[WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/analysis.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/balancesheet.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/cashflow.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/companies.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/documents.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/profitandloss.xlsx'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/raw/prosandcons.xlsx')]

In [5]:
list(CLEAN_DIR.glob("*"))

[WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/analysis_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/balancesheet_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/cashflow_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/companies_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/profit_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/prosandcons.csv')]

<h1> ETL Actual Development </h1>
<h2> Build Extract Layer </h2>
<h3> Generic Excel Loader Function</h3>

In [6]:
def load_excel_file(file_path, header_row=1):

    """
    Generic function to load Excel files
    """

    df = pd.read_excel(file_path, header=header_row)

    print(f"Loaded: {file_path.name}")

    print(f"Shape: {df.shape}")

    print("-" * 50)

    return df

In [7]:
# Test Extraction
companies_raw = load_excel_file(
    RAW_DIR / "companies.xlsx"
)

companies_raw.head()

Loaded: companies.xlsx
Shape: (92, 12)
--------------------------------------------------


,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10


In [8]:
# Load Multiple Files Together
raw_files = {

    "companies": "companies.xlsx",

    "balancesheet": "balancesheet.xlsx",

    "cashflow": "cashflow.xlsx",

    "analysis": "analysis.xlsx",

    "profitandloss": "profitandloss.xlsx",

    "prosandcons": "prosandcons.xlsx"
}

In [9]:
# Automated Multi-File Extraction
# Load All Raw Files Automatically
all_raw_data = {}

for table_name, file_name in raw_files.items():

    file_path = RAW_DIR / file_name

    try:
        df = load_excel_file(file_path)

        all_raw_data[table_name] = df

    except Exception as e:

        print(f"Error loading {file_name}")

        print(e)

        print("-" * 50)

Loaded: companies.xlsx
Shape: (92, 12)
--------------------------------------------------
Loaded: balancesheet.xlsx
Shape: (1312, 13)
--------------------------------------------------
Loaded: cashflow.xlsx
Shape: (1187, 7)
--------------------------------------------------
Loaded: analysis.xlsx
Shape: (20, 6)
--------------------------------------------------
Loaded: profitandloss.xlsx
Shape: (1276, 15)
--------------------------------------------------
Loaded: prosandcons.xlsx
Shape: (15, 4)
--------------------------------------------------


In [10]:
# Verify Loaded Tables
all_raw_data.keys()

dict_keys(['companies', 'balancesheet', 'cashflow', 'analysis', 'profitandloss', 'prosandcons'])

In [11]:
# Preview One Table Dynamically
all_raw_data["balancesheet"].head()

,id,company_id,year,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,136,ABB,Dec 2012,21.0,626,0,260,907,109,1,0,798,907
1,137,ABB,Mar 2014,21.0,767,0,351,1139,98,1,0,1040,1139
2,138,ABB,Mar 2015,21.0,916,0,436,1374,96,4,0,1274,1374
3,139,ABB,Mar 2016,21.0,1174,0,421,1616,108,3,0,1505,1616
4,140,ABB,Mar 2017,21.0,1366,0,679,2066,110,6,0,1950,2066


In [12]:
# Shape Summary Table
for table_name, df in all_raw_data.items():

    print(f"{table_name} --> {df.shape}")

companies --> (92, 12)
balancesheet --> (1312, 13)
cashflow --> (1187, 7)
analysis --> (20, 6)
profitandloss --> (1276, 15)
prosandcons --> (15, 4)


In [13]:
# Transform Functions
# Clean Text Nulls
def clean_null_values(df):
    """
    Replace string NULL values with actual NaN.
    """
    df = df.copy()
    
    df = df.replace(['NULL', 'Null', 'null', ''], np.nan)
    
    return df

In [14]:
# Year Standardization Function
def standardize_fiscal_year(year_value):
    """
    Extract fiscal year from values like:
    Mar 2024, Mar-24, Dec 2012, TTM
    """
    if pd.isna(year_value):
        return np.nan
    
    year_text = str(year_value).strip()
    
    if year_text.upper() == "TTM":
        return np.nan
    
    # Case: Mar 2024, Dec 2012
    match_4_digit = re.search(r'(\d{4})', year_text)
    if match_4_digit:
        return int(match_4_digit.group(1))
    
    # Case: Mar-13, Mar-24
    match_2_digit = re.search(r'(\d{2})$', year_text)
    if match_2_digit:
        return 2000 + int(match_2_digit.group(1))
    
    return np.nan

In [15]:
# Quick Test of Year Function
test_years = ['Mar 2024', 'Mar-13', 'Dec 2012', 'Sep 2024', 'TTM']

for y in test_years:
    print(y, "->", standardize_fiscal_year(y))

Mar 2024 -> 2024
Mar-13 -> 2013
Dec 2012 -> 2012
Sep 2024 -> 2024
TTM -> nan


In [16]:
# Numeric Conversion Helper
def convert_to_numeric(df, columns):
    """
    Convert selected columns to numeric safely.
    """
    df = df.copy()
    
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

In [17]:
# Test Helpers on Cashflow
test_cashflow = all_raw_data["cashflow"].copy()

test_cashflow = clean_null_values(test_cashflow)

test_cashflow['fiscal_year'] = test_cashflow['year'].apply(standardize_fiscal_year)

test_cashflow[['year', 'fiscal_year']].head(10)

,year,fiscal_year
0,Mar-13,2013
1,Mar-14,2014
2,Mar-15,2015
3,Mar-16,2016
4,Mar-17,2017
5,Mar-18,2018
6,Mar-19,2019
7,Mar-20,2020
8,Mar-21,2021
9,Mar-22,2022


In [18]:
# Transform Analysis Function
def clean_row_data(val):
    """
    Extract period and percentage value from text like:
    '10 Years: 21%', 'TTM: 43%', 'Last Year: 12%'
    """
    if pd.isna(val):
        return None, np.nan
    
    text = str(val).strip()
    
    match = re.search(r'^(.*?):\s*(-?\d+)', text)
    
    if match:
        period_raw = match.group(1).strip()
        value = float(match.group(2))
        
        if '10' in period_raw:
            period = '10Y'
        elif '5' in period_raw:
            period = '5Y'
        elif '3' in period_raw:
            period = '3Y'
        elif 'TTM' in period_raw.upper():
            period = 'TTM'
        elif '1' in period_raw or 'LAST' in period_raw.upper():
            period = '1Y'
        else:
            period = period_raw
        
        return period, value
    
    num_only = re.search(r'(-?\d+)', text)
    return None, float(num_only.group(1)) if num_only else np.nan

In [19]:
# Analysis Transform Function
def transform_analysis(df):
    df = clean_null_values(df)
    
    cleaned_data = []

    for _, row in df.iterrows():
        period, sales_growth = clean_row_data(row['compounded_sales_growth'])
        _, profit_growth = clean_row_data(row['compounded_profit_growth'])
        _, stock_cagr = clean_row_data(row['stock_price_cagr'])
        _, roe = clean_row_data(row['roe'])

        cleaned_data.append({
            'symbol': row['company_id'],
            'period': period,
            'sales_growth': sales_growth,
            'profit_growth': profit_growth,
            'stock_cagr': stock_cagr,
            'roe': roe
        })

    return pd.DataFrame(cleaned_data)

In [20]:
# Test Analysis Transform
analysis_transformed = transform_analysis(all_raw_data["analysis"])

analysis_transformed.head(20)

,symbol,period,sales_growth,profit_growth,stock_cagr,roe
0,HDFCBANK,10Y,21.0,22.0,15.0,17.0
1,SBILIFE,5Y,24.0,6.0,8.0,5.0
2,SBILIFE,3Y,17.0,9.0,7.0,13.0
3,SBILIFE,TTM,43.0,18.0,-2.0,12.0
4,TCS,10Y,11.0,9.0,14.0,40.0
5,TCS,5Y,10.0,8.0,16.0,44.0
6,TCS,3Y,14.0,12.0,8.0,47.0
7,TCS,TTM,5.0,8.0,16.0,52.0
8,WIPRO,10Y,8.0,3.0,12.0,18.0
9,WIPRO,5Y,9.0,4.0,20.0,17.0


In [21]:
analysis_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   symbol         20 non-null     str    
 1   period         20 non-null     str    
 2   sales_growth   20 non-null     float64
 3   profit_growth  20 non-null     float64
 4   stock_cagr     20 non-null     float64
 5   roe            20 non-null     float64
dtypes: float64(4), str(2)
memory usage: 1.1 KB


In [23]:
# Transform Companies Table
# Companies Transform Function
def transform_companies(df):

    df = clean_null_values(df)

    df = df.copy()

    # Rename columns
    df = df.rename(columns={

        'id': 'symbol',

        'nse_profile': 'nse_url',

        'bse_profile': 'bse_url',

        'roce_percentage': 'roce',

        'roe_percentage': 'roe'
    })

    # Clean company names
    df['company_name'] = (
        df['company_name']
        .astype(str)
        .str.strip()
        .str.replace(r'[\r\n]+', ' ', regex=True)
    )

    # Add sector columns
    df['sector'] = np.nan
    df['sub_sector'] = np.nan

    return df

In [24]:
# Test Companies Transform
companies_transformed = transform_companies(
    all_raw_data["companies"]
)

companies_transformed.head()

,symbol,company_logo,company_name,chart_link,about_company,website,nse_url,bse_url,face_value,book_value,roce,roe,sector,sub_sector
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90,NaN,NaN
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59,NaN,NaN
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64,NaN,NaN
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70,NaN,NaN
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10,NaN,NaN


In [25]:
companies_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   symbol         92 non-null     str    
 1   company_logo   91 non-null     str    
 2   company_name   92 non-null     str    
 3   chart_link     92 non-null     str    
 4   about_company  92 non-null     str    
 5   website        91 non-null     str    
 6   nse_url        91 non-null     str    
 7   bse_url        91 non-null     str    
 8   face_value     91 non-null     float64
 9   book_value     91 non-null     float64
 10  roce           91 non-null     float64
 11  roe            90 non-null     float64
 12  sector         0 non-null      float64
 13  sub_sector     0 non-null      float64
dtypes: float64(6), str(8)
memory usage: 10.2 KB


In [26]:
# Fix sector/sub_sector dtype
companies_transformed['sector'] = companies_transformed['sector'].astype('object')
companies_transformed['sub_sector'] = companies_transformed['sub_sector'].astype('object')

companies_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   symbol         92 non-null     str    
 1   company_logo   91 non-null     str    
 2   company_name   92 non-null     str    
 3   chart_link     92 non-null     str    
 4   about_company  92 non-null     str    
 5   website        91 non-null     str    
 6   nse_url        91 non-null     str    
 7   bse_url        91 non-null     str    
 8   face_value     91 non-null     float64
 9   book_value     91 non-null     float64
 10  roce           91 non-null     float64
 11  roe            90 non-null     float64
 12  sector         0 non-null      object 
 13  sub_sector     0 non-null      object 
dtypes: float64(4), object(2), str(8)
memory usage: 10.2+ KB


In [27]:
# Transform Balance Sheet Function
def transform_balancesheet(df):
    df = clean_null_values(df)
    df = df.copy()

    df = df.rename(columns={
        'company_id': 'symbol',
        'other_asset': 'other_assets'
    })

    df['fiscal_year'] = df['year'].apply(standardize_fiscal_year)

    numeric_cols = [
        'equity_capital', 'reserves', 'borrowings', 'other_liabilities',
        'total_liabilities', 'fixed_assets', 'cwip', 'investments',
        'other_assets', 'total_assets'
    ]

    df = convert_to_numeric(df, numeric_cols)

    df['debt_to_equity'] = df['borrowings'] / (
        df['equity_capital'] + df['reserves']
    )

    return df

In [28]:
# Test Balance Sheet Transform
balancesheet_transformed = transform_balancesheet(all_raw_data["balancesheet"])

balancesheet_transformed.head()

,id,symbol,year,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_assets,total_assets,fiscal_year,debt_to_equity
0,136,ABB,Dec 2012,21.0,626,0,260,907,109,1,0,798,907,2012,0.0
1,137,ABB,Mar 2014,21.0,767,0,351,1139,98,1,0,1040,1139,2014,0.0
2,138,ABB,Mar 2015,21.0,916,0,436,1374,96,4,0,1274,1374,2015,0.0
3,139,ABB,Mar 2016,21.0,1174,0,421,1616,108,3,0,1505,1616,2016,0.0
4,140,ABB,Mar 2017,21.0,1366,0,679,2066,110,6,0,1950,2066,2017,0.0


In [29]:
balancesheet_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1312 entries, 0 to 1311
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1312 non-null   int64  
 1   symbol             1312 non-null   str    
 2   year               1312 non-null   str    
 3   equity_capital     1312 non-null   float64
 4   reserves           1312 non-null   int64  
 5   borrowings         1312 non-null   int64  
 6   other_liabilities  1312 non-null   int64  
 7   total_liabilities  1312 non-null   int64  
 8   fixed_assets       1312 non-null   int64  
 9   cwip               1312 non-null   int64  
 10  investments        1312 non-null   int64  
 11  other_assets       1312 non-null   int64  
 12  total_assets       1312 non-null   int64  
 13  fiscal_year        1312 non-null   int64  
 14  debt_to_equity     1312 non-null   float64
dtypes: float64(2), int64(11), str(2)
memory usage: 153.9 KB


In [30]:
# Transform Cashflow Function
def transform_cashflow(df):
    df = clean_null_values(df)
    df = df.copy()

    df = df.rename(columns={
        'company_id': 'symbol'
    })

    df['fiscal_year'] = df['year'].apply(standardize_fiscal_year)

    numeric_cols = [
        'operating_activity',
        'investing_activity',
        'financing_activity',
        'net_cash_flow'
    ]

    df = convert_to_numeric(df, numeric_cols)

    df['free_cash_flow'] = df['operating_activity'] + df['investing_activity']

    df = df.dropna(subset=[
        'operating_activity',
        'investing_activity',
        'financing_activity',
        'net_cash_flow',
        'free_cash_flow'
    ])

    df = df.reset_index(drop=True)

    return df

In [31]:
# Test Cashflow Transform
cashflow_transformed = transform_cashflow(all_raw_data["cashflow"])

cashflow_transformed.head()

,id,symbol,year,operating_activity,investing_activity,financing_activity,net_cash_flow,fiscal_year,free_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0,2013,5577.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0,2014,5299.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0,2015,17562.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0,2016,14099.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0,2017,8328.0


In [32]:
cashflow_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1185 entries, 0 to 1184
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  1185 non-null   int64  
 1   symbol              1185 non-null   str    
 2   year                1185 non-null   str    
 3   operating_activity  1185 non-null   float64
 4   investing_activity  1185 non-null   float64
 5   financing_activity  1185 non-null   float64
 6   net_cash_flow       1185 non-null   float64
 7   fiscal_year         1185 non-null   int64  
 8   free_cash_flow      1185 non-null   float64
dtypes: float64(5), int64(2), str(2)
memory usage: 83.4 KB


In [33]:
# Transform Profit & Loss Table
# Transform Profit & Loss Function
def transform_profitloss(df):

    df = clean_null_values(df)

    df = df.copy()

    # Rename columns
    df = df.rename(columns={
        'company_id': 'symbol'
    })

    # Fiscal year extraction
    df['fiscal_year'] = df['year'].apply(
        standardize_fiscal_year
    )

    # Numeric columns
    numeric_cols = [

        'sales',
        'expenses',
        'operating_profit',
        'opm_percent',
        'other_income',
        'interest',
        'depreciation',
        'profit_before_tax',
        'tax_percent',
        'net_profit',
        'eps_in_rs',
        'dividend_payout_percent'
    ]

    df = convert_to_numeric(df, numeric_cols)

    # Computed metrics

    df['net_profit_margin_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['net_profit'] / df['sales']) * 100
    )

    df['expense_ratio_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['expenses'] / df['sales']) * 100
    )

    df['interest_coverage'] = np.where(

        df['interest'] == 0,

        np.nan,

        df['operating_profit'] / df['interest']
    )

    return df

In [34]:
# Test Profit Transform
profit_transformed = transform_profitloss(
    all_raw_data["profitandloss"]
)

profit_transformed.head()

,id,symbol,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,profit_before_tax,tax_percentage,net_profit,eps,dividend_payout,fiscal_year,net_profit_margin_pct,expense_ratio_pct,interest_coverage
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,215,33.0,145,68.0,25.0,2012.0,8.771930,87.779794,NaN
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,295,33.0,198,93.0,25.0,2014.0,8.699473,88.268893,NaN
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,344,34.0,229,108.0,29.0,2015.0,10.004369,86.369594,NaN
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,398,36.0,255,120.0,29.0,2016.0,9.755164,86.074981,121.666667
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,436,37.0,277,130.0,31.0,2017.0,9.541853,86.290045,199.000000


In [35]:
profit_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1276 entries, 0 to 1275
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     1276 non-null   int64  
 1   symbol                 1276 non-null   str    
 2   year                   1276 non-null   str    
 3   sales                  1276 non-null   int64  
 4   expenses               1276 non-null   int64  
 5   operating_profit       1263 non-null   float64
 6   opm_percentage         1261 non-null   float64
 7   other_income           1276 non-null   int64  
 8   interest               1276 non-null   int64  
 9   depreciation           1276 non-null   int64  
 10  profit_before_tax      1276 non-null   int64  
 11  tax_percentage         1181 non-null   float64
 12  net_profit             1276 non-null   int64  
 13  eps                    1271 non-null   float64
 14  dividend_payout        1173 non-null   float64
 15  fiscal_year    

In [36]:
# Quick Metric Preview
profit_transformed[[
    'symbol',
    'sales',
    'net_profit',
    'net_profit_margin_pct',
    'expense_ratio_pct',
    'interest_coverage'
]].head(10)

,symbol,sales,net_profit,net_profit_margin_pct,expense_ratio_pct,interest_coverage
0,ABB,1653,145,8.771930,87.779794,NaN
1,ABB,2276,198,8.699473,88.268893,NaN
2,ABB,2289,229,10.004369,86.369594,NaN
3,ABB,2614,255,9.755164,86.074981,121.666667
4,ABB,2903,277,9.541853,86.290045,199.000000
5,ABB,3298,401,12.158884,84.111583,131.250000
6,ABB,3679,450,12.231585,83.555314,302.500000
7,ABB,4093,593,14.488151,81.505009,84.111111
8,ABB,4310,691,16.032483,78.607889,51.222222
9,ABB,4913,799,16.262976,77.997150,56.947368


In [37]:
# Correct Profit Transform Function
def transform_profitloss(df):

    df = clean_null_values(df)

    df = df.copy()

    # Rename columns
    df = df.rename(columns={
        'company_id': 'symbol'
    })

    # Clean year text
    df['year'] = (
        df['year']
        .astype(str)
        .str.strip()
    )

    # Fiscal year extraction
    df['fiscal_year'] = df['year'].apply(
        standardize_fiscal_year
    )

    # Numeric conversion
    numeric_cols = [

        'sales',
        'expenses',
        'operating_profit',
        'opm_percentage',
        'other_income',
        'interest',
        'depreciation',
        'profit_before_tax',
        'tax_percentage',
        'net_profit',
        'eps',
        'dividend_payout'
    ]

    df = convert_to_numeric(df, numeric_cols)

    # Computed metrics

    df['net_profit_margin_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['net_profit'] / df['sales']) * 100
    )

    df['expense_ratio_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['expenses'] / df['sales']) * 100
    )

    df['interest_coverage'] = np.where(

        df['interest'] == 0,

        np.nan,

        df['operating_profit'] / df['interest']
    )

    return df

In [38]:
# Recreate Profit Transform
profit_transformed = transform_profitloss(
    all_raw_data["profitandloss"]
)

profit_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1276 entries, 0 to 1275
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     1276 non-null   int64  
 1   symbol                 1276 non-null   str    
 2   year                   1276 non-null   str    
 3   sales                  1276 non-null   int64  
 4   expenses               1276 non-null   int64  
 5   operating_profit       1263 non-null   float64
 6   opm_percentage         1261 non-null   float64
 7   other_income           1276 non-null   int64  
 8   interest               1276 non-null   int64  
 9   depreciation           1276 non-null   int64  
 10  profit_before_tax      1276 non-null   int64  
 11  tax_percentage         1181 non-null   float64
 12  net_profit             1276 non-null   int64  
 13  eps                    1271 non-null   float64
 14  dividend_payout        1173 non-null   float64
 15  fiscal_year    

In [39]:
# Create Final Profit Clean Table
profit_clean_final = profit_transformed.copy()

# Keep only annual March records
profit_clean_final = profit_clean_final[
    profit_clean_final['year'].str.contains('Mar', na=False)
]

# Drop rows where fiscal_year is missing
profit_clean_final = profit_clean_final.dropna(subset=['fiscal_year'])

# Fiscal year integer
profit_clean_final['fiscal_year'] = profit_clean_final['fiscal_year'].astype(int)

# Final columns
profit_clean_final = profit_clean_final[[
    'symbol',
    'year',
    'fiscal_year',
    'sales',
    'expenses',
    'operating_profit',
    'opm_percentage',
    'net_profit',
    'eps',
    'dividend_payout',
    'net_profit_margin_pct',
    'expense_ratio_pct',
    'interest_coverage'
]]

profit_clean_final = profit_clean_final.reset_index(drop=True)

In [40]:
profit_clean_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1119 entries, 0 to 1118
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   symbol                 1119 non-null   str    
 1   year                   1119 non-null   str    
 2   fiscal_year            1119 non-null   int64  
 3   sales                  1119 non-null   int64  
 4   expenses               1119 non-null   int64  
 5   operating_profit       1107 non-null   float64
 6   opm_percentage         1106 non-null   float64
 7   net_profit             1119 non-null   int64  
 8   eps                    1116 non-null   float64
 9   dividend_payout        1115 non-null   float64
 10  net_profit_margin_pct  1118 non-null   float64
 11  expense_ratio_pct      1118 non-null   float64
 12  interest_coverage      1078 non-null   float64
dtypes: float64(7), int64(4), str(2)
memory usage: 113.8 KB


In [41]:
profit_clean_final.head()

,symbol,year,fiscal_year,sales,expenses,operating_profit,opm_percentage,net_profit,eps,dividend_payout,net_profit_margin_pct,expense_ratio_pct,interest_coverage
0,ABB,Mar 2014,2014,2276,2009,267.0,12.0,198,93.0,25.0,8.699473,88.268893,NaN
1,ABB,Mar 2015,2015,2289,1977,312.0,14.0,229,108.0,29.0,10.004369,86.369594,NaN
2,ABB,Mar 2016,2016,2614,2250,365.0,14.0,255,120.0,29.0,9.755164,86.074981,121.666667
3,ABB,Mar 2017,2017,2903,2505,398.0,14.0,277,130.0,31.0,9.541853,86.290045,199.000000
4,ABB,Mar 2018,2018,3298,2774,525.0,16.0,401,189.0,29.0,12.158884,84.111583,131.250000


In [44]:
# Transform Pros & Cons
# Transform Pros & Cons Function
def transform_prosandcons(df):

    df = clean_null_values(df)

    df = df.copy()

    df = df.rename(columns={
        'company_id': 'symbol'
    })

    cleaned_rows = []

    for _, row in df.iterrows():

        # PROS
        for col in ['pros_1', 'pros_2', 'pros_3', 'pros_4', 'pros_5']:

            if col in df.columns:

                val = row.get(col)

                if pd.notna(val):

                    cleaned_rows.append({

                        'symbol': row['symbol'],

                        'is_pro': True,

                        'insight_text': str(val).strip(),

                        'source': 'MANUAL',

                        'confidence': 1.0
                    })

        # CONS
        for col in ['cons_1', 'cons_2', 'cons_3', 'cons_4', 'cons_5']:

            if col in df.columns:

                val = row.get(col)

                if pd.notna(val):

                    cleaned_rows.append({

                        'symbol': row['symbol'],

                        'is_pro': False,

                        'insight_text': str(val).strip(),

                        'source': 'MANUAL',

                        'confidence': 1.0
                    })

    return pd.DataFrame(cleaned_rows)

In [48]:
# Test Pros & Cons Transform
proscons_transformed = transform_prosandcons(
    all_raw_data["prosandcons"]
)

proscons_transformed.head(15)

""


In [49]:
proscons_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [50]:
all_raw_data["prosandcons"].columns

Index([                                              1,
                                            'HDFCBANK',
            'Company is expected to give good quarter',
       'Stock is trading at 2.76 times its book value'],
      dtype='object')

In [51]:
# FIX MAJOUR MISTAKE
# Reload current prosandcons
prosandcons_raw = pd.read_excel(
    RAW_DIR / "prosandcons.xlsx",
    header=0
)

prosandcons_raw.head()

,id,company_id,pros,cons
0,1,HDFCBANK,Company is expected to give good quarter,Stock is trading at 2.76 times its book value
1,2,HDFCBANK,Company has delivered good profit growth of 23...,Company has low interest coverage ratio.
2,3,HDFCBANK,Company has been maintaining a healthy dividen...,"Contingent liabilities of Rs.24,09,821 Cr."
3,4,HDFCBANK,Company's median sales growth is 16.4% of last...,"Earnings include an other income of Rs.1,55,87..."
4,5,SBILIFE,Company is almost debt free.,Stock is trading at 9.11 times its book value


In [52]:
prosandcons_raw.columns

Index(['id', 'company_id', 'pros', 'cons'], dtype='str')

In [53]:
# Update dictionary
all_raw_data["prosandcons"] = prosandcons_raw

In [54]:
# Correct transform function
def transform_prosandcons(df):
    df = clean_null_values(df)
    df = df.copy()

    df = df.rename(columns={
        'company_': 'symbol',
        'company': 'symbol',
        'company_id': 'symbol'
    })

    cleaned_rows = []

    for _, row in df.iterrows():

        if pd.notna(row.get('pros')):
            cleaned_rows.append({
                'symbol': row['symbol'],
                'is_pro': True,
                'category': 'GENERAL',
                'insight_text': str(row['pros']).strip(),
                'source': 'MANUAL',
                'confidence': 1.0
            })

        if pd.notna(row.get('cons')):
            cleaned_rows.append({
                'symbol': row['symbol'],
                'is_pro': False,
                'category': 'GENERAL',
                'insight_text': str(row['cons']).strip(),
                'source': 'MANUAL',
                'confidence': 1.0
            })

    return pd.DataFrame(cleaned_rows)

In [55]:
# Cross checks
proscons_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [56]:
proscons_transformed['is_pro'].value_counts()

KeyError: 'is_pro'

In [57]:
# FIX MAJOUR MISTAKE AGAIN
# Correct transform function
def transform_prosandcons(df):
    df = clean_null_values(df)
    df = df.copy()

    # column cleanup
    df.columns = df.columns.astype(str).str.strip()

    # company column detect
    if 'company_id' in df.columns:
        company_col = 'company_id'
    elif 'company_' in df.columns:
        company_col = 'company_'
    elif 'company' in df.columns:
        company_col = 'company'
    else:
        raise ValueError("Company column not found")

    cleaned_rows = []

    for _, row in df.iterrows():

        symbol = row[company_col]

        if pd.notna(row.get('pros')):
            cleaned_rows.append({
                'symbol': symbol,
                'is_pro': True,
                'category': 'GENERAL',
                'insight_text': str(row['pros']).strip(),
                'source': 'MANUAL',
                'confidence': 1.0
            })

        if pd.notna(row.get('cons')):
            cleaned_rows.append({
                'symbol': symbol,
                'is_pro': False,
                'category': 'GENERAL',
                'insight_text': str(row['cons']).strip(),
                'source': 'MANUAL',
                'confidence': 1.0
            })

    return pd.DataFrame(cleaned_rows)

In [58]:
proscons_transformed = transform_prosandcons(prosandcons_raw)

proscons_transformed.head(15)

,symbol,is_pro,category,insight_text,source,confidence
0,HDFCBANK,True,GENERAL,Company is expected to give good quarter,MANUAL,1.0
1,HDFCBANK,False,GENERAL,Stock is trading at 2.76 times its book value,MANUAL,1.0
2,HDFCBANK,True,GENERAL,Company has delivered good profit growth of 23...,MANUAL,1.0
3,HDFCBANK,False,GENERAL,Company has low interest coverage ratio.,MANUAL,1.0
4,HDFCBANK,True,GENERAL,Company has been maintaining a healthy dividen...,MANUAL,1.0
5,HDFCBANK,False,GENERAL,"Contingent liabilities of Rs.24,09,821 Cr.",MANUAL,1.0
6,HDFCBANK,True,GENERAL,Company's median sales growth is 16.4% of last...,MANUAL,1.0
7,HDFCBANK,False,GENERAL,"Earnings include an other income of Rs.1,55,87...",MANUAL,1.0
8,SBILIFE,True,GENERAL,Company is almost debt free.,MANUAL,1.0
9,SBILIFE,False,GENERAL,Stock is trading at 9.11 times its book value,MANUAL,1.0


In [59]:
proscons_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   symbol        26 non-null     str    
 1   is_pro        26 non-null     bool   
 2   category      26 non-null     str    
 3   insight_text  26 non-null     str    
 4   source        26 non-null     str    
 5   confidence    26 non-null     float64
dtypes: bool(1), float64(1), str(4)
memory usage: 1.2 KB


In [60]:
proscons_transformed['is_pro'].value_counts()

is_pro
False    15
True     11
Name: count, dtype: int64

In [61]:
# AGAIN Update all_raw_data
all_raw_data["prosandcons"] = prosandcons_raw

In [62]:
# Verify dictionary update
all_raw_data["prosandcons"].head()

,id,company_id,pros,cons
0,1,HDFCBANK,Company is expected to give good quarter,Stock is trading at 2.76 times its book value
1,2,HDFCBANK,Company has delivered good profit growth of 23...,Company has low interest coverage ratio.
2,3,HDFCBANK,Company has been maintaining a healthy dividen...,"Contingent liabilities of Rs.24,09,821 Cr."
3,4,HDFCBANK,Company's median sales growth is 16.4% of last...,"Earnings include an other income of Rs.1,55,87..."
4,5,SBILIFE,Company is almost debt free.,Stock is trading at 9.11 times its book value


In [63]:
# Optional final test from dictionary
proscons_transformed = transform_prosandcons(
    all_raw_data["prosandcons"]
)

proscons_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   symbol        26 non-null     str    
 1   is_pro        26 non-null     bool   
 2   category      26 non-null     str    
 3   insight_text  26 non-null     str    
 4   source        26 non-null     str    
 5   confidence    26 non-null     float64
dtypes: bool(1), float64(1), str(4)
memory usage: 1.2 KB


In [64]:
# Save All Transformed Files
# Save cleaned transformed outputs
analysis_transformed.to_csv(CLEAN_DIR / "analysis_clean.csv", index=False)

companies_transformed.to_csv(CLEAN_DIR / "companies_clean.csv", index=False)

balancesheet_transformed.to_csv(CLEAN_DIR / "balancesheet_clean.csv", index=False)

cashflow_transformed.to_csv(CLEAN_DIR / "cashflow_clean.csv", index=False)

profit_clean_final.to_csv(CLEAN_DIR / "profit_clean.csv", index=False)

proscons_transformed.to_csv(CLEAN_DIR / "prosandcons_clean.csv", index=False)

print("All transformed files saved successfully.")

All transformed files saved successfully.


In [65]:
# Verify saved files
list(CLEAN_DIR.glob("*"))

[WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/analysis_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/balancesheet_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/cashflow_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/companies_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/profit_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/prosandcons.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/prosandcons_clean.csv')]

In [66]:
# Clean Files Shape Audit
clean_files = list(CLEAN_DIR.glob("*.csv"))

for file in clean_files:
    df = pd.read_csv(file)
    print(file.name, "->", df.shape)

analysis_clean.csv -> (20, 6)
balancesheet_clean.csv -> (1312, 15)
cashflow_clean.csv -> (1185, 9)
companies_clean.csv -> (92, 14)
profit_clean.csv -> (1119, 13)
prosandcons.csv -> (16, 4)
prosandcons_clean.csv -> (26, 6)


<h2> Create Master ETL Function </h2>

In [67]:
def run_etl_pipeline():
    """
    Complete ETL pipeline:
    1. Load raw Excel files
    2. Transform each dataset
    3. Save clean CSV files
    """

    print("Starting ETL Pipeline...")
    print("-" * 60)

    # 1. Extract
    raw_data = {}

    for table_name, file_name in raw_files.items():
        file_path = RAW_DIR / file_name
        raw_data[table_name] = load_excel_file(file_path)

    print("Extraction completed.")
    print("-" * 60)

    # 2. Special reload for prosandcons because header is already row 0
    raw_data["prosandcons"] = pd.read_excel(
        RAW_DIR / "prosandcons.xlsx",
        header=0
    )

    # 3. Transform
    transformed_data = {
        "analysis_clean": transform_analysis(raw_data["analysis"]),
        "companies_clean": transform_companies(raw_data["companies"]),
        "balancesheet_clean": transform_balancesheet(raw_data["balancesheet"]),
        "cashflow_clean": transform_cashflow(raw_data["cashflow"]),
        "profit_clean": transform_profitloss(raw_data["profitandloss"]),
        "prosandcons_clean": transform_prosandcons(raw_data["prosandcons"])
    }

    # 4. Final profit filter
    profit_final = transformed_data["profit_clean"].copy()

    profit_final = profit_final[
        profit_final["year"].str.contains("Mar", na=False)
    ]

    profit_final = profit_final.dropna(subset=["fiscal_year"])
    profit_final["fiscal_year"] = profit_final["fiscal_year"].astype(int)

    profit_final = profit_final[[
        "symbol",
        "year",
        "fiscal_year",
        "sales",
        "expenses",
        "operating_profit",
        "opm_percentage",
        "net_profit",
        "eps",
        "dividend_payout",
        "net_profit_margin_pct",
        "expense_ratio_pct",
        "interest_coverage"
    ]]

    profit_final = profit_final.reset_index(drop=True)

    transformed_data["profit_clean"] = profit_final

    print("Transformation completed.")
    print("-" * 60)

    # 5. Save
    for name, df in transformed_data.items():
        output_path = CLEAN_DIR / f"{name}.csv"
        df.to_csv(output_path, index=False)
        print(f"Saved: {output_path.name} -> {df.shape}")

    print("-" * 60)
    print("ETL Pipeline completed successfully.")

    return transformed_data

In [68]:
# Run Master ETL Pipeline
etl_outputs = run_etl_pipeline()

Starting ETL Pipeline...
------------------------------------------------------------
Loaded: companies.xlsx
Shape: (92, 12)
--------------------------------------------------
Loaded: balancesheet.xlsx
Shape: (1312, 13)
--------------------------------------------------
Loaded: cashflow.xlsx
Shape: (1187, 7)
--------------------------------------------------
Loaded: analysis.xlsx
Shape: (20, 6)
--------------------------------------------------
Loaded: profitandloss.xlsx
Shape: (1276, 15)
--------------------------------------------------
Loaded: prosandcons.xlsx
Shape: (15, 4)
--------------------------------------------------
Extraction completed.
------------------------------------------------------------
Transformation completed.
------------------------------------------------------------
Saved: analysis_clean.csv -> (20, 6)
Saved: companies_clean.csv -> (92, 14)
Saved: balancesheet_clean.csv -> (1312, 15)
Saved: cashflow_clean.csv -> (1185, 9)
Saved: profit_clean.csv -> (1119, 1

In [69]:
# Cross Check ETL Outputs
for name, df in etl_outputs.items():
    print(name, "->", df.shape)

analysis_clean -> (20, 6)
companies_clean -> (92, 14)
balancesheet_clean -> (1312, 15)
cashflow_clean -> (1185, 9)
profit_clean -> (1119, 13)
prosandcons_clean -> (26, 6)


<h2> Create Script Export Block </h2>

In [70]:
script_content = """
# ============================================
# NIFTY100 ETL CLEAN & TRANSFORM PIPELINE
# ============================================

import pandas as pd
import numpy as np
import re

from pathlib import Path


# ============================================
# PATHS
# ============================================

BASE_DIR = Path(r"C:/Users/LENOVO/Desktop/nifty100_project")

RAW_DIR = BASE_DIR / "data" / "raw"

CLEAN_DIR = BASE_DIR / "data" / "clean"


# ============================================
# HELPER FUNCTIONS
# ============================================

def clean_null_values(df):

    df = df.copy()

    df = df.replace(
        ['NULL', 'Null', 'null', ''],
        np.nan
    )

    return df


def standardize_fiscal_year(year_value):

    if pd.isna(year_value):
        return np.nan

    year_text = str(year_value).strip()

    if year_text.upper() == "TTM":
        return np.nan

    match_4_digit = re.search(r'(\\d{4})', year_text)

    if match_4_digit:
        return int(match_4_digit.group(1))

    match_2_digit = re.search(r'(\\d{2})$', year_text)

    if match_2_digit:
        return 2000 + int(match_2_digit.group(1))

    return np.nan


def convert_to_numeric(df, columns):

    df = df.copy()

    for col in columns:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors='coerce'
            )

    return df
"""

In [71]:
# Write Script To File
script_path = ETL_DIR / "02_clean_transform.py"

with open(script_path, "w", encoding="utf-8") as file:

    file.write(script_content)

print("Script file created successfully.")

print(script_path)

Script file created successfully.
C:\Users\LENOVO\Desktop\nifty100_project\etl\02_clean_transform.py


<h2> Create Transform Function Block </h2>

In [72]:
transform_functions = """

# ============================================
# TRANSFORM FUNCTIONS
# ============================================

def transform_analysis(df):

    df = clean_null_values(df)

    cleaned_data = []

    for _, row in df.iterrows():

        period, sales_growth = clean_row_data(
            row['compounded_sales_growth']
        )

        _, profit_growth = clean_row_data(
            row['compounded_profit_growth']
        )

        _, stock_cagr = clean_row_data(
            row['stock_price_cagr']
        )

        _, roe = clean_row_data(
            row['roe']
        )

        cleaned_data.append({

            'symbol': row['company_id'],

            'period': period,

            'sales_growth': sales_growth,

            'profit_growth': profit_growth,

            'stock_cagr': stock_cagr,

            'roe': roe
        })

    return pd.DataFrame(cleaned_data)


def transform_companies(df):

    df = clean_null_values(df)

    df = df.copy()

    df = df.rename(columns={

        'id': 'symbol',

        'nse_profile': 'nse_url',

        'bse_profile': 'bse_url',

        'roce_percentage': 'roce',

        'roe_percentage': 'roe'
    })

    df['company_name'] = (

        df['company_name']
        .astype(str)
        .str.strip()
        .str.replace(r'[\\r\\n]+', ' ', regex=True)
    )

    df['sector'] = np.nan

    df['sub_sector'] = np.nan

    return df


def transform_balancesheet(df):

    df = clean_null_values(df)

    df = df.copy()

    df = df.rename(columns={

        'company_id': 'symbol',

        'other_asset': 'other_assets'
    })

    df['fiscal_year'] = df['year'].apply(
        standardize_fiscal_year
    )

    numeric_cols = [

        'equity_capital',
        'reserves',
        'borrowings',
        'other_liabilities',
        'total_liabilities',
        'fixed_assets',
        'cwip',
        'investments',
        'other_assets',
        'total_assets'
    ]

    df = convert_to_numeric(df, numeric_cols)

    df['debt_to_equity'] = (

        df['borrowings'] /

        (
            df['equity_capital'] +
            df['reserves']
        )
    )

    return df
"""

In [73]:
# Append Into Script
with open(script_path, "a", encoding="utf-8") as file:

    file.write(transform_functions)

print("Transform functions appended successfully.")

Transform functions appended successfully.


<h2> Append Remaining Helper + Transform Functions </h2>
<h3> Create Remaining Function Block</h3>

In [74]:
remaining_functions = """

def clean_row_data(val):

    if pd.isna(val):
        return None, np.nan

    text = str(val).strip()

    match = re.search(r'^(.*?):\\s*(-?\\d+)', text)

    if match:

        period_raw = match.group(1).strip()

        value = float(match.group(2))

        if '10' in period_raw:
            period = '10Y'

        elif '5' in period_raw:
            period = '5Y'

        elif '3' in period_raw:
            period = '3Y'

        elif 'TTM' in period_raw.upper():
            period = 'TTM'

        elif '1' in period_raw:
            period = '1Y'

        else:
            period = period_raw

        return period, value

    num_only = re.search(r'(-?\\d+)', text)

    return None, float(num_only.group(1)) if num_only else np.nan


def transform_cashflow(df):

    df = clean_null_values(df)

    df = df.copy()

    df = df.rename(columns={
        'company_id': 'symbol'
    })

    df['fiscal_year'] = df['year'].apply(
        standardize_fiscal_year
    )

    numeric_cols = [

        'operating_activity',
        'investing_activity',
        'financing_activity',
        'net_cash_flow'
    ]

    df = convert_to_numeric(df, numeric_cols)

    df['free_cash_flow'] = (

        df['operating_activity'] +

        df['investing_activity']
    )

    return df


def transform_profitloss(df):

    df = clean_null_values(df)

    df = df.copy()

    df = df.rename(columns={
        'company_id': 'symbol'
    })

    df['year'] = (
        df['year']
        .astype(str)
        .str.strip()
    )

    df['fiscal_year'] = df['year'].apply(
        standardize_fiscal_year
    )

    numeric_cols = [

        'sales',
        'expenses',
        'operating_profit',
        'opm_percentage',
        'other_income',
        'interest',
        'depreciation',
        'profit_before_tax',
        'tax_percentage',
        'net_profit',
        'eps',
        'dividend_payout'
    ]

    df = convert_to_numeric(df, numeric_cols)

    df['net_profit_margin_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['net_profit'] / df['sales']) * 100
    )

    df['expense_ratio_pct'] = np.where(

        df['sales'] == 0,

        np.nan,

        (df['expenses'] / df['sales']) * 100
    )

    df['interest_coverage'] = np.where(

        df['interest'] == 0,

        np.nan,

        df['operating_profit'] / df['interest']
    )

    return df


def transform_prosandcons(df):

    df = clean_null_values(df)

    df = df.copy()

    df.columns = df.columns.astype(str).str.strip()

    if 'company_id' in df.columns:
        company_col = 'company_id'

    elif 'company_' in df.columns:
        company_col = 'company_'

    elif 'company' in df.columns:
        company_col = 'company'

    else:
        raise ValueError("Company column not found")

    cleaned_rows = []

    for _, row in df.iterrows():

        symbol = row[company_col]

        if pd.notna(row.get('pros')):

            cleaned_rows.append({

                'symbol': symbol,

                'is_pro': True,

                'category': 'GENERAL',

                'insight_text': str(row['pros']).strip(),

                'source': 'MANUAL',

                'confidence': 1.0
            })

        if pd.notna(row.get('cons')):

            cleaned_rows.append({

                'symbol': symbol,

                'is_pro': False,

                'category': 'GENERAL',

                'insight_text': str(row['cons']).strip(),

                'source': 'MANUAL',

                'confidence': 1.0
            })

    return pd.DataFrame(cleaned_rows)
"""

In [75]:
# Append Remaining Functions
with open(script_path, "a", encoding="utf-8") as file:

    file.write(remaining_functions)

print("Remaining functions appended successfully.")

Remaining functions appended successfully.


<h2> Add Master Pipeline Block </h2>

In [76]:
pipeline_block = """

# ============================================
# MASTER ETL PIPELINE
# ============================================

def load_excel_file(file_path, header_row=1):

    df = pd.read_excel(file_path, header=header_row)

    print(f"Loaded: {file_path.name} -> {df.shape}")

    return df


def run_etl_pipeline():

    print("Starting Nifty100 ETL Pipeline...")
    print("-" * 60)

    raw_files = {

        "companies": "companies.xlsx",

        "balancesheet": "balancesheet.xlsx",

        "cashflow": "cashflow.xlsx",

        "analysis": "analysis.xlsx",

        "profitandloss": "profitandloss.xlsx"
    }

    raw_data = {}

    for table_name, file_name in raw_files.items():

        raw_data[table_name] = load_excel_file(
            RAW_DIR / file_name,
            header_row=1
        )

    # prosandcons current file has header at row 0
    raw_data["prosandcons"] = pd.read_excel(
        RAW_DIR / "prosandcons.xlsx",
        header=0
    )

    print("Extraction completed.")
    print("-" * 60)

    transformed_data = {

        "analysis_clean": transform_analysis(raw_data["analysis"]),

        "companies_clean": transform_companies(raw_data["companies"]),

        "balancesheet_clean": transform_balancesheet(raw_data["balancesheet"]),

        "cashflow_clean": transform_cashflow(raw_data["cashflow"]),

        "prosandcons_clean": transform_prosandcons(raw_data["prosandcons"])
    }

    profit_full = transform_profitloss(raw_data["profitandloss"])

    profit_clean = profit_full[
        profit_full["year"].str.contains("Mar", na=False)
    ]

    profit_clean = profit_clean.dropna(subset=["fiscal_year"])

    profit_clean["fiscal_year"] = profit_clean["fiscal_year"].astype(int)

    profit_clean = profit_clean[[

        "symbol",
        "year",
        "fiscal_year",
        "sales",
        "expenses",
        "operating_profit",
        "opm_percentage",
        "net_profit",
        "eps",
        "dividend_payout",
        "net_profit_margin_pct",
        "expense_ratio_pct",
        "interest_coverage"
    ]]

    profit_clean = profit_clean.reset_index(drop=True)

    transformed_data["profit_clean"] = profit_clean

    print("Transformation completed.")
    print("-" * 60)

    for name, df in transformed_data.items():

        output_path = CLEAN_DIR / f"{name}.csv"

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path.name} -> {df.shape}")

    print("-" * 60)
    print("ETL Pipeline completed successfully.")

    return transformed_data


if __name__ == "__main__":

    run_etl_pipeline()
"""

In [77]:
# Append Pipeline Block
with open(script_path, "a", encoding="utf-8") as file:

    file.write(pipeline_block)

print("Master pipeline block appended successfully.")

Master pipeline block appended successfully.


In [78]:
!python "C:\Users\LENOVO\Desktop\nifty100_project\etl\02_clean_transform.py"

Starting Nifty100 ETL Pipeline...
------------------------------------------------------------
Loaded: companies.xlsx -> (92, 12)
Loaded: balancesheet.xlsx -> (1312, 13)
Loaded: cashflow.xlsx -> (1187, 7)
Loaded: analysis.xlsx -> (20, 6)
Loaded: profitandloss.xlsx -> (1276, 15)
Extraction completed.
------------------------------------------------------------
Transformation completed.
------------------------------------------------------------
Saved: analysis_clean.csv -> (20, 6)
Saved: companies_clean.csv -> (92, 14)
Saved: balancesheet_clean.csv -> (1312, 15)
Saved: cashflow_clean.csv -> (1187, 9)
Saved: prosandcons_clean.csv -> (26, 6)
Saved: profit_clean.csv -> (1119, 13)
------------------------------------------------------------
ETL Pipeline completed successfully.


In [79]:
# Fix transform_cashflow() inside .py
script_text = script_path.read_text(encoding="utf-8")

old_cashflow_block = """
    df['free_cash_flow'] = (

        df['operating_activity'] +

        df['investing_activity']
    )

    return df
"""

new_cashflow_block = """
    df['free_cash_flow'] = (

        df['operating_activity'] +

        df['investing_activity']
    )

    df = df.dropna(subset=[

        'operating_activity',
        'investing_activity',
        'financing_activity',
        'net_cash_flow',
        'free_cash_flow'
    ])

    df = df.reset_index(drop=True)

    return df
"""

script_text = script_text.replace(old_cashflow_block, new_cashflow_block)

script_path.write_text(script_text, encoding="utf-8")

print("Cashflow transform fixed successfully.")

Cashflow transform fixed successfully.


In [80]:
# Re-run ETL script
!python "C:\Users\LENOVO\Desktop\nifty100_project\etl\02_clean_transform.py"

Starting Nifty100 ETL Pipeline...
------------------------------------------------------------
Loaded: companies.xlsx -> (92, 12)
Loaded: balancesheet.xlsx -> (1312, 13)
Loaded: cashflow.xlsx -> (1187, 7)
Loaded: analysis.xlsx -> (20, 6)
Loaded: profitandloss.xlsx -> (1276, 15)
Extraction completed.
------------------------------------------------------------
Transformation completed.
------------------------------------------------------------
Saved: analysis_clean.csv -> (20, 6)
Saved: companies_clean.csv -> (92, 14)
Saved: balancesheet_clean.csv -> (1312, 15)
Saved: cashflow_clean.csv -> (1185, 9)
Saved: prosandcons_clean.csv -> (26, 6)
Saved: profit_clean.csv -> (1119, 13)
------------------------------------------------------------
ETL Pipeline completed successfully.


In [81]:
# Final clean file audit
for file in CLEAN_DIR.glob("*.csv"):
    df = pd.read_csv(file)
    print(file.name, "->", df.shape)

analysis_clean.csv -> (20, 6)
balancesheet_clean.csv -> (1312, 15)
cashflow_clean.csv -> (1185, 9)
companies_clean.csv -> (92, 14)
profit_clean.csv -> (1119, 13)
prosandcons.csv -> (16, 4)
prosandcons_clean.csv -> (26, 6)


<h3>NOW Create 01_extract.py </h3>
<h4> Create Extract Script Content </h4>

In [82]:
extract_script = """

# ============================================
# NIFTY100 EXTRACTION LAYER
# ============================================

import pandas as pd

from pathlib import Path


# ============================================
# PATHS
# ============================================

BASE_DIR = Path(r"C:/Users/LENOVO/Desktop/nifty100_project")

RAW_DIR = BASE_DIR / "data" / "raw"


# ============================================
# RAW FILE CONFIG
# ============================================

RAW_FILES = {

    "companies": "companies.xlsx",

    "balancesheet": "balancesheet.xlsx",

    "cashflow": "cashflow.xlsx",

    "analysis": "analysis.xlsx",

    "profitandloss": "profitandloss.xlsx",

    "prosandcons": "prosandcons.xlsx"
}


# ============================================
# EXTRACTION FUNCTION
# ============================================

def load_excel_file(file_path, header_row=1):

    df = pd.read_excel(
        file_path,
        header=header_row
    )

    print(f"Loaded: {file_path.name} -> {df.shape}")

    return df


# ============================================
# EXTRACTION PIPELINE
# ============================================

def run_extraction_pipeline():

    print("Starting Extraction Pipeline...")
    print("-" * 60)

    raw_data = {}

    for table_name, file_name in RAW_FILES.items():

        file_path = RAW_DIR / file_name

        # prosandcons special case
        if table_name == "prosandcons":

            df = pd.read_excel(
                file_path,
                header=0
            )

        else:

            df = load_excel_file(
                file_path,
                header_row=1
            )

        raw_data[table_name] = df

    print("-" * 60)

    print("Extraction completed successfully.")

    return raw_data


if __name__ == "__main__":

    run_extraction_pipeline()

"""

In [83]:
# Save Extract Script
extract_script_path = ETL_DIR / "01_extract.py"

with open(extract_script_path, "w", encoding="utf-8") as file:

    file.write(extract_script)

print("01_extract.py created successfully.")

print(extract_script_path)

01_extract.py created successfully.
C:\Users\LENOVO\Desktop\nifty100_project\etl\01_extract.py


In [84]:
# Run Extract Script
!python "C:\Users\LENOVO\Desktop\nifty100_project\etl\01_extract.py"

Starting Extraction Pipeline...
------------------------------------------------------------
Loaded: companies.xlsx -> (92, 12)
Loaded: balancesheet.xlsx -> (1312, 13)
Loaded: cashflow.xlsx -> (1187, 7)
Loaded: analysis.xlsx -> (20, 6)
Loaded: profitandloss.xlsx -> (1276, 15)
------------------------------------------------------------
Extraction completed successfully.


<h3> Create 03_load_to_postgres.py </h3>
<h4> Warehouse Loading Script </h4>
<h4> Create PostgreSQL Loader Script </h4>

In [85]:
load_script = """

# ============================================
# NIFTY100 POSTGRES LOADER
# ============================================

import pandas as pd

from pathlib import Path

from sqlalchemy import create_engine


# ============================================
# PATHS
# ============================================

BASE_DIR = Path(r"C:/Users/LENOVO/Desktop/nifty100_project")

CLEAN_DIR = BASE_DIR / "data" / "clean"


# ============================================
# DATABASE CONFIG
# ============================================

DB_USER = "postgres"

DB_PASSWORD = "YOUR_PASSWORD"

DB_HOST = "localhost"

DB_PORT = "5432"

DB_NAME = "nifty100_analysis"


# ============================================
# CREATE ENGINE
# ============================================

engine = create_engine(

    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


# ============================================
# CLEAN FILE CONFIG
# ============================================

clean_files = {

    "analysis_clean": "analysis_clean.csv",

    "companies_clean": "companies_clean.csv",

    "balancesheet_clean": "balancesheet_clean.csv",

    "cashflow_clean": "cashflow_clean.csv",

    "profit_clean": "profit_clean.csv",

    "prosandcons_clean": "prosandcons_clean.csv"
}


# ============================================
# LOAD VALIDATION
# ============================================

def validate_clean_files():

    print("Validating clean files...")
    print("-" * 60)

    for table_name, file_name in clean_files.items():

        file_path = CLEAN_DIR / file_name

        df = pd.read_csv(file_path)

        print(f"{file_name} -> {df.shape}")

    print("-" * 60)

    print("Validation completed successfully.")


if __name__ == "__main__":

    validate_clean_files()

"""

In [86]:
# Save Loader Script
load_script_path = ETL_DIR / "03_load_to_postgres.py"

with open(load_script_path, "w", encoding="utf-8") as file:

    file.write(load_script)

print("03_load_to_postgres.py created successfully.")

print(load_script_path)

03_load_to_postgres.py created successfully.
C:\Users\LENOVO\Desktop\nifty100_project\etl\03_load_to_postgres.py


In [87]:
# Run Loader Validation Script
!python "C:\Users\LENOVO\Desktop\nifty100_project\etl\03_load_to_postgres.py"

Traceback (most recent call last):
  File "C:\Users\LENOVO\Desktop\nifty100_project\etl\03_load_to_postgres.py", line 11, in <module>
    from sqlalchemy import create_engine
ModuleNotFoundError: No module named 'sqlalchemy'


In [88]:
!pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.3 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 1.4 MB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.4 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.4 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.3 MB/s  0:00:01
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [89]:
# Again Run Loader Validation Script
!python "C:\Users\LENOVO\Desktop\nifty100_project\etl\03_load_to_postgres.py"

Validating clean files...
------------------------------------------------------------
analysis_clean.csv -> (20, 6)
companies_clean.csv -> (92, 14)
balancesheet_clean.csv -> (1312, 15)
cashflow_clean.csv -> (1185, 9)
profit_clean.csv -> (1119, 13)
prosandcons_clean.csv -> (26, 6)
------------------------------------------------------------
Validation completed successfully.


In [90]:
# Final ETL Folder Audit
list(ETL_DIR.glob("*"))

[WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/etl/01_extract.py'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/etl/02_clean_transform.py'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/etl/03_load_to_postgres.py')]

In [91]:
# Clean Folder Audit
list(CLEAN_DIR.glob("*"))

[WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/analysis_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/balancesheet_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/cashflow_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/companies_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/profit_clean.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/prosandcons.csv'),
 WindowsPath('C:/Users/LENOVO/Desktop/nifty100_project/data/clean/prosandcons_clean.csv')]

<h2> Final Project Status Summary </h2>

In [92]:
print("NIFTY100 DATA ENGINEERING PIPELINE STATUS")
print("=" * 60)

print("\\nRAW DATA LAYER")
print("-" * 30)
print("Raw Excel files available.")

print("\\nETL LAYER")
print("-" * 30)
print("01_extract.py -> READY")
print("02_clean_transform.py -> READY")
print("03_load_to_postgres.py -> READY")

print("\\nCLEAN DATA LAYER")
print("-" * 30)
print("Clean transformed CSV files generated.")

print("\\nWAREHOUSE LAYER")
print("-" * 30)
print("PostgreSQL warehouse created successfully.")

print("\\nPROJECT STATUS")
print("-" * 30)
print("ETL + Warehouse Pipeline COMPLETED SUCCESSFULLY.")

NIFTY100 DATA ENGINEERING PIPELINE STATUS
\nRAW DATA LAYER
------------------------------
Raw Excel files available.
\nETL LAYER
------------------------------
01_extract.py -> READY
02_clean_transform.py -> READY
03_load_to_postgres.py -> READY
\nCLEAN DATA LAYER
------------------------------
Clean transformed CSV files generated.
\nWAREHOUSE LAYER
------------------------------
PostgreSQL warehouse created successfully.
\nPROJECT STATUS
------------------------------
ETL + Warehouse Pipeline COMPLETED SUCCESSFULLY.
